# Publish Model to HF Hub

Select a model checkpoint from W&B artifacts and publish it as the official
`kaya-go/moku-v2` model on Hugging Face Hub.

**Workflow:**

1. List available W&B model artifacts (with mAP, epoch, run info)
2. Set the artifact path to publish
3. Download and load the model
4. Quick sanity check on real test set
5. Push to HF Hub

## Available Artifacts

Browse model artifacts from W&B and pick the one to publish.

In [6]:
%load_ext autoreload
%autoreload 2

from dotenv import load_dotenv

load_dotenv()

import numpy as np
import pandas as pd
from datasets import load_dataset

from moku.dataset import CATEGORIES, ID_TO_CATEGORY
from moku.evaluation import evaluate_cd_ap, evaluate_map
from moku.runs import load_model_from_wandb
from moku.model import make_eval_transform

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


## Select Artifact

Set the artifact path below. Use the `name` column from the table above.
Append `:latest` or `:vN` for a specific version.

In [3]:
# ← Set the artifact to publish
ARTIFACT = "model-r10_os3_lr3e-4_cosmin100 :latest"

HF_MODEL = "kaya-go/moku-v3"
HF_MODEL_BASELINE = "kaya-go/moku-v2"
HF_DATASET = "kaya-go/moku-v3"

print(f"Will publish: {ARTIFACT}")
print(f"Target: {HF_MODEL}")
print(f"Baseline: {HF_MODEL_BASELINE}")
print(f"Dataset: {HF_DATASET}")

Will publish: model-r10_os3_lr3e-4_cosmin100 :latest
Target: kaya-go/moku-v3
Baseline: kaya-go/moku-v2
Dataset: kaya-go/moku-v3


## Download & Load Model

In [4]:
ip, model = load_model_from_wandb(ARTIFACT)

total = sum(p.numel() for p in model.parameters())
print(f"Model loaded: {total:,} parameters")
print(f"Labels: {model.config.id2label}")

wandb: Downloading large artifact 'model-r10_os3_lr3e-4_cosmin100 :latest', 76.78MB. 3 files...
wandb:   3 of 3 files downloaded.  
Done. 00:00:00.3 (306.1MB/s)


Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

Model loaded: 20,075,740 parameters
Labels: {0: 'black_stone', 1: 'white_stone', 2: 'board_corner'}


## Sanity Check — Metrics on Test Set

Quick evaluation on the test split to confirm the model performs as expected.

| Metric | What it measures |
|--------|------------------|
| **mAP** | Overall detection quality (COCO IoU 0.50:0.95) |
| **mAP@50** | Overall detection quality (COCO IoU≥0.50) |
| **corner_R4** | Top-4 corner recall — matches inference (always keep top-4) |
| **stone_cdAP@2%** | Macro center-distance AP for `black_stone` + `white_stone` |

In [7]:
from transformers import RTDetrForObjectDetection, RTDetrImageProcessor

ds = load_dataset(HF_DATASET)

# ── Helper to evaluate a single model ──
def _eval_model(m, processor, ds_test):
    ds_test.set_transform(make_eval_transform(processor))
    map_metrics = evaluate_map(model=m, dataset=ds_test, image_processor=processor, batch_size=8)
    cd = evaluate_cd_ap(model=m, dataset=ds_test, image_processor=processor, batch_size=8)
    corner_R4 = cd["corner_R4"]
    stone_cdAP = np.mean([cd["per_class"][c]["cdAP@2%"] for c in ("black_stone", "white_stone")])
    return {
        "mAP": map_metrics.get("map", float("nan")),
        "mAP@50": map_metrics.get("map_50", float("nan")),
        "corner_R4": corner_R4,
        "stone_cdAP@2%": stone_cdAP,
    }

test_split = ds["test"]

# ── Evaluate candidate (W&B artifact) ──
print(f"Evaluating candidate: {ARTIFACT}")
candidate_metrics = _eval_model(model, ip, test_split)

# ── Evaluate current HF Hub model (baseline) ──
print(f"Evaluating baseline: {HF_MODEL_BASELINE} (current HF Hub)")
ip_baseline = RTDetrImageProcessor.from_pretrained(HF_MODEL_BASELINE)
model_baseline = RTDetrForObjectDetection.from_pretrained(
    HF_MODEL_BASELINE,
    num_labels=len(CATEGORIES),
    id2label=ID_TO_CATEGORY,
    label2id=CATEGORIES,
)
model_baseline.eval()
baseline_metrics = _eval_model(model_baseline, ip_baseline, test_split)

# ── Comparison table ──
comparison = pd.DataFrame([
    {"model": f"baseline ({HF_MODEL_BASELINE})", **baseline_metrics},
    {"model": f"candidate ({ARTIFACT})", **candidate_metrics},
])
comparison = comparison.set_index("model")

# Add delta row
delta = comparison.iloc[1] - comparison.iloc[0]
delta.name = "Δ (candidate − baseline)"
comparison = pd.concat([comparison, delta.to_frame().T])

display(comparison)

Evaluating candidate: model-r10_os3_lr3e-4_cosmin100 :latest
Evaluating baseline: kaya-go/moku-v2 (current HF Hub)


Loading weights:   0%|          | 0/526 [00:00<?, ?it/s]

,mAP,mAP@50,corner_R4,stone_cdAP@2%
baseline (kaya-go/moku-v2),0.401283,0.724238,0.810,0.93285
candidate (model-r10_os3_lr3e-4_cosmin100 :latest),0.493767,0.862200,0.825,0.91870
Δ (candidate − baseline),0.092485,0.137962,0.015,-0.01415


## Push to HF Hub

Publish the model and image processor to the official HF model repository.

**Warning**: This overwrites the current model on the `main` branch of `kaya-go/moku-v3`.

In [8]:
# Uncomment to push (destructive — overwrites current model on HF Hub)
model.push_to_hub(HF_MODEL)
ip.push_to_hub(HF_MODEL)
print(f"Model pushed to https://huggingface.co/{HF_MODEL}")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Processing Files (0 / 0): |          |  0.00B /  0.00B            

New Data Upload: |          |  0.00B /  0.00B            

README.md: 0.00B [00:00, ?B/s]

Model pushed to https://huggingface.co/kaya-go/moku-v3
